# 11 — Group Relative Policy Optimization with Verifiable Decisions

## Objective

Explain group-relative advantages, unit-test an exact-answer verifier, and run a bounded Group Relative Policy Optimization (GRPO)-style update from the shared Supervised Fine-Tuning (SFT) checkpoint on the frozen PubMedQA training partition.

## Reward design

One point is awarded for a correct exact decision label; an additional 0.05 point is awarded for a valid one-word answer. A wrong but valid answer receives 0.05. A verbose or unparseable answer receives zero. The correctness component dominates formatting.

$$A_i=\frac{r_i-\bar r}{\sigma_r+\epsilon}$$

Here $r_i$ is a completion's reward, $\bar r$ and $\sigma_r$ are its group mean and standard deviation, and $\epsilon$ prevents division by zero. Identical group rewards give zero advantages. Training reward is not independent evidence of generalization; notebook 12 must evaluate frozen validation IDs. Notebook 09's learned reward model is not used.

### Define a verifiable reward function with role-aware parsing

In [1]:
import re
import numpy as np


def completion_text(completion):
    """Read generated text, ignoring any non-assistant conversational messages."""
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list):
        if any(not isinstance(message, dict) for message in completion):
            raise TypeError("Expected conversational completion messages to be dictionaries")
        return "\n".join(str(message.get("content", "")) for message in completion
                         if message.get("role") == "assistant")
    raise TypeError(f"Unsupported completion type: {type(completion).__name__}")


def exact_answer(text):
    match = re.fullmatch(r"(?i)\s*(yes|no|maybe)\s*[.]?\s*", text)
    return match.group(1).lower() if match else "invalid"


def answer_reward(completions, final_decision, **kwargs):
    """Return one deterministic scalar reward per generated assistant completion."""
    if len(completions) != len(final_decision):
        raise ValueError("Completion/label alignment differs: reward must not be misassigned")
    if any(str(expected).strip().lower() not in ("yes", "no", "maybe") for expected in final_decision):
        raise ValueError("Expected labels must be yes, no, or maybe")
    scores = []
    for completion, expected in zip(completions, final_decision):
        parsed = exact_answer(completion_text(completion))
        correctness = float(parsed == str(expected).strip().lower())
        formatting = 0.05 * float(parsed != "invalid")
        scores.append(correctness + formatting)
    return scores


unit_rewards = answer_reward(
    completions=["yes", "maybe", "not sure", [{"role": "assistant", "content": "no"}]],
    final_decision=["yes", "no", "maybe", "no"],
)
assert unit_rewards == [1.05, 0.05, 0.0, 1.05]
assert answer_reward([[{"role": "user", "content": "yes"},
                       {"role": "assistant", "content": "no"}]], ["no"]) == [1.05]
print("Illustrative verified rewards:", unit_rewards)
values = np.array(unit_rewards, dtype=float)
advantages = (values - values.mean()) / (values.std() + 1e-8)
print("Illustrative group-relative advantages:", np.round(advantages, 3).tolist())
assert abs(float(advantages.mean())) < 1e-6
assert np.all((np.ones(4) - np.ones(4).mean()) / (np.ones(4).std() + 1e-8) == 0)


Illustrative verified rewards: [1.05, 0.05, 0.0, 1.05]
Illustrative group-relative advantages: [0.999, -0.951, -1.048, 0.999]


### Data partition and shared adapter requirements

In [2]:
# Locate the research workspace from the repository root or the notebooks folder.
from pathlib import Path
cwd = Path.cwd().resolve()
if (cwd / "exploration" / "notebooks").is_dir():
    RESEARCH = cwd / "exploration"
elif cwd.name == "notebooks" and cwd.parent.name == "exploration":
    RESEARCH = cwd.parent
elif cwd.name == "exploration" and (cwd / "notebooks").is_dir():
    RESEARCH = cwd
else:
    raise FileNotFoundError("Open Jupyter from the repository root, exploration/, or exploration/notebooks/.")

RUNS = RESEARCH / "runs"
RUNS.mkdir(parents=True, exist_ok=True)
print("Research workspace:", RESEARCH)
print("Experiment outputs:", RUNS)

Research workspace: C:\xuexi\Anaconda\BioLLM-PostTraining\exploration
Experiment outputs: C:\xuexi\Anaconda\BioLLM-PostTraining\exploration\runs


In [3]:
# PubMedQA dataset protocol: the same source, group assignment and prompt data
# must be used in notebooks 03, 04, 05, 06, 11, 12 and 18.
import hashlib
import json
import os
from collections import Counter

DATASET_NAME = "qiaojin/PubMedQA"
DATASET_CONFIG = "pqa_labeled"      # Explicitly labeled PubMedQA questions.
SEED = 42
LABELS = ("yes", "no", "maybe")
MANIFEST = RESEARCH / "data" / "pubmedqa_split_manifest.json"
MANIFEST.parent.mkdir(parents=True, exist_ok=True)


def paper_id(row):
    """Stable publication ID used to prevent paper-level train/evaluation overlap."""
    value = str(row.get("pubid", "")).strip()
    if not value:
        raise ValueError("Missing PubMedQA publication ID; cannot split safely.")
    return value


def evidence_text(row):
    """Extract abstract sentences from PubMedQA's nested context object."""
    context = row.get("context") or {}
    sentences = context.get("contexts", []) if isinstance(context, dict) else []
    if not isinstance(sentences, list):
        raise TypeError("Expected context['contexts'] to be a list of abstract sentences.")
    return "\n".join(sentence.strip() for sentence in sentences if isinstance(sentence, str))


def assign_split(identifier):
    """Deterministically assign ALL rows with one publication ID to the same split."""
    digest = hashlib.sha256(f"{SEED}:{identifier}".encode("utf-8")).digest()
    fraction = int.from_bytes(digest[:8], "big") / 2**64
    if fraction < 0.80:
        return "train"
    if fraction < 0.90:
        return "validation"
    return "test"


def source_fingerprint(rows):
    """Hash canonical source fields to detect modifications between experiments."""
    canonical = sorted(
        (paper_id(row), str(row["question"]), evidence_text(row), str(row["final_decision"]).lower())
        for row in rows
    )
    payload = json.dumps(canonical, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def prepare_partitions(rows, create_manifest=False):
    """Check labels, record stable splits and require identical upstream source data."""
    records = list(rows)
    identifiers = [paper_id(row) for row in records]
    if len(identifiers) != len(set(identifiers)):
        raise ValueError("Repeated publication IDs: review source records before evaluation.")
    if any(str(row["final_decision"]).lower() not in LABELS for row in records):
        raise ValueError("Unexpected label; expected one of yes/no/maybe.")
    if any(not evidence_text(row) for row in records):
        raise ValueError("Empty evidence context; review source-data schema.")

    partitions = {name: [] for name in ("train", "validation", "test")}
    for row in records:
        partitions[assign_split(paper_id(row))].append(row)
    metadata = {
        "dataset": DATASET_NAME,
        "config": DATASET_CONFIG,
        "split_algorithm": "SHA256(seed:pubid), buckets 0-0.8/0.8-0.9/0.9-1.0",
        "seed": SEED,
        "fingerprint": source_fingerprint(records),
        "ids": {name: sorted(paper_id(row) for row in subset) for name, subset in partitions.items()},
    }
    if MANIFEST.exists():
        previous = json.loads(MANIFEST.read_text(encoding="utf-8"))
        if previous != metadata:
            raise RuntimeError("Dataset differs from saved split manifest. Do not silently overwrite it; inspect data revision and protocol.")
    elif create_manifest:
        MANIFEST.write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
        print("Created reproducible split manifest:", MANIFEST)
    else:
        raise FileNotFoundError("Split manifest missing. Run notebook 03 with RUN_PUBLIC_DATA=1 first.")

    for name, subset in partitions.items():
        print(f"{name}: n={len(subset)}, labels={dict(Counter(str(r['final_decision']).lower() for r in subset))}")
    if min(len(p) for p in partitions.values()) == 0:
        raise ValueError("At least one split is empty; check dataset size and split rules.")
    return partitions

In [4]:
SYSTEM_INSTRUCTION = (
    "Answer the biomedical question using ONLY the supplied abstract evidence. "
    "Choose exactly one decision label: yes, no, or maybe. "
    "Return the label alone, without an explanation. "
    "Use maybe when the evidence does not establish yes or no."
)


def question_messages(row):
    """Use one consistent role-aware input protocol across model variants."""
    full_text = evidence_text(row)
    if not full_text:
        raise ValueError("Question has no evidence passage.")
    # Keep both the abstract introduction and conclusion; the conclusion often
    # carries the decision evidence. Apply the same limit in training/evaluation.
    if len(full_text) > 1600:
        text = full_text[:950] + "\n[Middle of abstract omitted for length]\n" + full_text[-600:]
    else:
        text = full_text
    return [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": f"Question: {row['question']}\n\nAbstract evidence:\n{text}"},
    ]


def parse_decision(generated_text):
    """Parse an exact one-label answer; preserve raw outputs when parsing fails."""
    import re
    text = str(generated_text).strip()
    match = re.fullmatch(r"(?i)(yes|no|maybe)[.\s]*", text)
    return match.group(1).lower() if match else "invalid"

### Bounded GRPO-style training

The installed TRL GRPO trainer cannot import with PyTorch 2.4.1 because `DTensor` is absent, and the local four-bit model-loading path is incompatible. This notebook implements the group-normalized advantage, per-token clipped importance-ratio surrogate, and frozen SFT-reference KL penalty directly with the existing unquantized bfloat16 Low-Rank Adaptation (LoRA) stack. Four sampled completions share each PubMedQA training prompt. Groups with identical rewards have zero advantages and are skipped. The old policy is fixed within each two-epoch rollout update. This short run measures optimizer mechanics and training rewards; it does not establish validation improvement or complete biomedical reasoning.

In [5]:
import os
import json

if os.getenv("RUN_TRAINING", "0") == "1":
    import time
    import random
    import numpy as np
    import torch
    import torch.nn.functional as F
    import transformers, peft, datasets
    from datasets import Dataset, config, load_dataset
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel

    model_id = os.getenv("MODEL_ID", "Qwen/Qwen2.5-0.5B-Instruct")
    sft_dir = RUNS / "sft_adapter"
    sft_metadata = json.loads((sft_dir / "run_manifest.json").read_text(encoding="utf-8"))
    split_manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))
    if sft_metadata["model_id"] != model_id or sft_metadata["dataset_fingerprint"] != split_manifest["fingerprint"]:
        raise ValueError("SFT model or PubMedQA source differs from the frozen lineage")
    revision = sft_metadata["model_revision"]
    if not revision:
        raise ValueError("SFT base-model revision is missing")
    if os.getenv("HF_DATASETS_OFFLINE", "0") == "1":
        files = list(Path(config.HF_DATASETS_CACHE).glob(
            "qiaojin___pub_med_qa/pqa_labeled/*/*/pub_med_qa-train.arrow"))
        if len(files) != 1:
            raise FileNotFoundError(f"Expected one cached PubMedQA Arrow file, found {len(files)}")
        source = Dataset.from_file(str(files[0]))
        source_location = str(files[0])
    else:
        source = load_dataset(DATASET_NAME, DATASET_CONFIG, split="train")
        source_location = "Hugging Face Datasets load_dataset"
    partitions = prepare_partitions(source, create_manifest=False)
    train_rows = sorted(partitions["train"], key=paper_id)
    if not set(paper_id(row) for row in train_rows) == set(split_manifest["ids"]["train"]):
        raise ValueError("Training IDs differ from the frozen partition")
    tokenizer = AutoTokenizer.from_pretrained(str(sft_dir), local_files_only=True)
    if tokenizer.chat_template is None:
        raise ValueError("SFT tokenizer lacks a chat template")
    tokenizer.pad_token = tokenizer.eos_token
    max_prompt_length = 768
    max_completion_length = 8
    eligible = []
    excluded_prompt_ids = []
    for row in train_rows:
        prompt_ids = tokenizer.apply_chat_template(
            question_messages(row), tokenize=True, add_generation_prompt=True).input_ids
        if len(prompt_ids) > max_prompt_length:
            excluded_prompt_ids.append(paper_id(row))
        else:
            eligible.append((row, prompt_ids))
    limit = int(os.getenv("GRPO_TRAIN_MAX_ITEMS", "80"))
    rollout_steps = int(os.getenv("GRPO_MAX_STEPS", "40"))
    groups_per_step = 2
    group_size = 4
    if limit < rollout_steps * groups_per_step or limit > len(eligible) or rollout_steps < 1:
        raise ValueError("Need at least two distinct eligible training prompts per rollout step")
    chosen = np.random.default_rng(SEED).choice(len(eligible), size=limit, replace=False)
    selected = [eligible[int(i)] for i in chosen]
    selected_ids = [paper_id(row) for row, _ in selected]
    if len(selected_ids) != len(set(selected_ids)) or not set(selected_ids).issubset(set(split_manifest["ids"]["train"])):
        raise ValueError("Selected GRPO prompts are not unique training IDs")
    print("Selected training prompts:", len(selected), "labels:",
          dict(Counter(str(row["final_decision"]).lower() for row, _ in selected)))
    print("Selected prompt token range:", min(len(ids) for _, ids in selected),
          max(len(ids) for _, ids in selected), "overlength source training prompts:", len(excluded_prompt_ids))

    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
        raise RuntimeError("This bounded run requires a bfloat16 CUDA GPU")
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    torch.cuda.reset_peak_memory_stats()
    start = time.perf_counter()
    def load_sft(trainable):
        base = AutoModelForCausalLM.from_pretrained(
            model_id, revision=revision, local_files_only=True, dtype=torch.bfloat16).to("cuda")
        wrapped = PeftModel.from_pretrained(base, str(sft_dir), is_trainable=trainable).to("cuda")
        wrapped.eval()  # Disable LoRA dropout while retaining gradients for the policy.
        wrapped.config.use_cache = False
        return wrapped
    policy = load_sft(True)
    reference = load_sft(False)
    reference.requires_grad_(False)
    optimizer = torch.optim.AdamW((p for p in policy.parameters() if p.requires_grad), lr=1e-5)
    beta = 0.01
    clip_epsilon = 0.2

    def completion_logprobs(model, full_ids, prefix_length):
        tokens = torch.tensor([full_ids], device="cuda")
        logits = model(input_ids=tokens, attention_mask=torch.ones_like(tokens), use_cache=False).logits.float()
        targets = tokens[:, prefix_length:]
        prediction_logits = logits[:, prefix_length - 1:prefix_length - 1 + targets.shape[1], :]
        return F.log_softmax(prediction_logits, dim=-1).gather(-1, targets.unsqueeze(-1)).squeeze(-1).reshape(-1)

    def generate_group(row, prompt_ids):
        prompt = torch.tensor([prompt_ids], device="cuda")
        with torch.inference_mode():
            sequences = policy.generate(
                input_ids=prompt, attention_mask=torch.ones_like(prompt),
                do_sample=True, temperature=1.0, top_p=1.0,
                num_return_sequences=group_size, max_new_tokens=max_completion_length,
                pad_token_id=tokenizer.eos_token_id,
            )
        completions = []
        for sequence in sequences:
            generated_ids = sequence[len(prompt_ids):].tolist()
            if tokenizer.eos_token_id in generated_ids:
                generated_ids = generated_ids[:generated_ids.index(tokenizer.eos_token_id) + 1]
            if not generated_ids:
                raise ValueError("Empty generated completion")
            text = tokenizer.decode(generated_ids, skip_special_tokens=True)
            completions.append({"full_ids": prompt_ids + generated_ids, "text": text})
        rewards = np.asarray(answer_reward(
            [entry["text"] for entry in completions],
            [str(row["final_decision"]).lower()] * group_size), dtype=float)
        if not np.all(np.isfinite(rewards)):
            raise FloatingPointError("Nonfinite verifiable rewards")
        std = float(rewards.std())
        advantages = (rewards - float(rewards.mean())) / (std + 1e-8) if std > 0 else np.zeros(group_size)
        with torch.inference_mode():
            for entry in completions:
                entry["old_logprobs"] = completion_logprobs(policy, entry["full_ids"], len(prompt_ids)).detach()
                entry["reference_logprobs"] = completion_logprobs(reference, entry["full_ids"], len(prompt_ids)).detach()
        return {"paper_id": paper_id(row), "label": str(row["final_decision"]).lower(),
                "rewards": rewards, "advantages": advantages, "completions": completions}

    logs = []
    optimizer_updates = 0
    for step in range(rollout_steps):
        batch = selected[step * groups_per_step:(step + 1) * groups_per_step]
        groups = [generate_group(row, ids) for row, ids in batch]
        active = [group for group in groups if np.any(group["advantages"] != 0)]
        epoch_losses, epoch_kl, epoch_clipped = [], [], []
        if active:
            # The old-policy log-probabilities stay fixed for both optimization epochs.
            for epoch in range(2):
                optimizer.zero_grad(set_to_none=True)
                for group in active:
                    for advantage, entry in zip(group["advantages"], group["completions"]):
                        current = completion_logprobs(policy, entry["full_ids"],
                                                      len(entry["full_ids"]) - len(entry["old_logprobs"]))
                        ratio = torch.exp(current - entry["old_logprobs"])
                        advantage_tensor = torch.tensor(float(advantage), device="cuda")
                        unclipped = ratio * advantage_tensor
                        clipped = ratio.clamp(1 - clip_epsilon, 1 + clip_epsilon) * advantage_tensor
                        ref_delta = entry["reference_logprobs"] - current
                        kl = torch.exp(ref_delta) - ref_delta - 1
                        loss = -(torch.minimum(unclipped, clipped) - beta * kl).mean()
                        if not torch.isfinite(loss):
                            raise FloatingPointError("Nonfinite group-relative loss")
                        (loss / (len(active) * group_size)).backward()
                        epoch_losses.append(float(loss.detach().item()))
                        epoch_kl.append(float(kl.detach().mean().item()))
                        epoch_clipped.append(float(((ratio < 1 - clip_epsilon) |
                                                    (ratio > 1 + clip_epsilon)).float().mean().item()))
                torch.nn.utils.clip_grad_norm_((p for p in policy.parameters() if p.requires_grad), 1.0)
                optimizer.step()
                optimizer_updates += 1
        all_rewards = np.concatenate([group["rewards"] for group in groups])
        all_text = [entry["text"] for group in groups for entry in group["completions"]]
        log = {"rollout_step": step + 1, "paper_ids": [group["paper_id"] for group in groups],
               "informative_groups": len(active), "uniform_groups": len(groups) - len(active),
               "mean_reward": float(all_rewards.mean()),
               "exact_accuracy": float(np.mean(all_rewards >= 1.0)),
               "valid_format": float(np.mean([exact_answer(text) != "invalid" for text in all_text])),
               "policy_loss": float(np.mean(epoch_losses)) if epoch_losses else None,
               "mean_kl_estimator": float(np.mean(epoch_kl)) if epoch_kl else None,
               "clip_fraction": float(np.mean(epoch_clipped)) if epoch_clipped else None,
               "optimizer_updates": optimizer_updates}
        logs.append(log)
        if (step + 1) % 4 == 0:
            print(log, flush=True)

    output_dir = RUNS / "grpo_adapter"
    output_dir.mkdir(exist_ok=True)
    policy.save_pretrained(str(output_dir), safe_serialization=True)
    tokenizer.save_pretrained(str(output_dir))
    details = {"variant": "grpo", "method": "explicit clipped group-relative policy objective, unquantized LoRA",
               "model_id": model_id, "model_revision": revision,
               "sft_source": str(sft_dir), "dataset": DATASET_NAME, "config": DATASET_CONFIG,
               "dataset_fingerprint": split_manifest["fingerprint"], "source_location": source_location,
               "partition": "train", "selected_ids": selected_ids,
               "used_rollout_ids": [identifier for log in logs for identifier in log["paper_ids"]],
               "sft_training_id_overlap": len(set(selected_ids) & set(sft_metadata["selected_ids"])),
               "excluded_overlength_train_ids": excluded_prompt_ids, "seed": SEED,
               "rollout_steps": rollout_steps, "completed_rollout_steps": len(logs),
               "groups_per_step": groups_per_step, "group_size": group_size,
               "optimization_epochs_per_rollout": 2, "optimizer_updates": optimizer_updates,
               "max_prompt_length": max_prompt_length, "max_completion_length": max_completion_length,
               "sampling": {"temperature": 1.0, "top_p": 1.0, "do_sample": True},
               "reward": "1.0 exact decision + 0.05 strict one-label format",
               "learning_rate": 1e-5, "beta_kl": beta, "clip_epsilon": clip_epsilon,
               "uniform_groups": sum(log["uniform_groups"] for log in logs),
               "informative_groups": sum(log["informative_groups"] for log in logs),
               "trainable_parameters": sum(p.numel() for p in policy.parameters() if p.requires_grad),
               "runtime_seconds": time.perf_counter() - start,
               "peak_gpu_gib": torch.cuda.max_memory_allocated() / 2**30,
               "versions": {"torch": torch.__version__, "transformers": transformers.__version__,
                            "peft": peft.__version__, "datasets": datasets.__version__},
               "run_status": "completed"}
    (output_dir / "run_manifest.json").write_text(json.dumps(details, indent=2) + "\n", encoding="utf-8")
    (output_dir / "log_history.json").write_text(json.dumps(logs, indent=2) + "\n", encoding="utf-8")
    print("Saved bounded GRPO-style adapter:", output_dir)
else:
    print("GRPO training disabled. Verifiable-reward unit checks remain available above.")


train: n=775, labels={'yes': 433, 'no': 255, 'maybe': 87}
validation: n=121, labels={'no': 40, 'yes': 64, 'maybe': 17}
test: n=104, labels={'yes': 55, 'no': 43, 'maybe': 6}


Selected training prompts: 80 labels: {'yes': 51, 'no': 21, 'maybe': 8}
Selected prompt token range: 190 541 overlength source training prompts: 0


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

C:\Users\weiju\anaconda3\envs\torch\lib\site-packages\transformers\integrations\sdpa_attention.py:92: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


...

{'rollout_step': 32, 'paper_ids': ['23810330', '24516646'], 'informative_groups': 1, 'uniform_groups': 1, 'mean_reward': 0.925, 'exact_accuracy': 0.875, 'valid_format': 1.0, 'policy_loss': -0.003941111266613007, 'mean_kl_estimator': 0.08576842397451401, 'clip_fraction': 0.0, 'optimizer_updates': 48}


{'rollout_step': 36, 'paper_ids': ['23806388', '11079675'], 'informative_groups': 1, 'uniform_groups': 1, 'mean_reward': 0.8, 'exact_accuracy': 0.75, 'valid_format': 1.0, 'policy_loss': -0.01476237177848816, 'mean_kl_estimator': 0.083765409886837, 'clip_fraction': 0.0, 'optimizer_updates': 54}


{'rollout_step': 40, 'paper_ids': ['26063028', '20736887'], 'informative_groups': 2, 'uniform_groups': 0, 'mean_reward': 0.5437500000000001, 'exact_accuracy': 0.5, 'valid_format': 0.875, 'policy_loss': 0.007973864674568176, 'mean_kl_estimator': 0.16160558257251978, 'clip_fraction': 0.0, 'optimizer_updates': 60}


Saved bounded GRPO-style adapter: C:\xuexi\Anaconda\BioLLM-PostTraining\exploration\runs\grpo_adapter


C:\Users\weiju\anaconda3\envs\torch\lib\site-packages\peft\utils\save_and_load.py:438: UserWarning: Could not find a config file in Qwen/Qwen2.5-0.5B-Instruct - will assume that the vocabulary was not modified.
  warnings.warn(


### Results

The GRPO-style run used 80 distinct PubMedQA training IDs over 40 rollout steps and 320 sampled completions. Exactly 185/320 completions matched source labels; 318/320 met the strict output format. Forty-two prompt groups varied in reward and 38 tied. The saved adapter later scored 76/121 on validation.

### Interpretation

Informative groups provided a relative training signal. Tied groups contributed no group-relative advantage.

### Limitations

The 320 completions are on-policy training samples, not independent evaluation. Validation had zero maybe predictions and lower macro-F1 than SFT; the small accuracy difference is uncertain.

### Key Observations

Training reward and output validity cannot substitute for classwise held-out accuracy.


## Key observations

Verifiable labels provide a narrow, deterministic training reward. Groups with identical answers yield zero group-relative learning signal, while a mixed group can update the policy. The clipped surrogate and reference-policy penalty limit incentives for large changes but do not guarantee correctness. Saved training rewards are not independent model-quality evidence; notebook 12 must compare the SFT and GRPO adapters on the frozen validation partition using the same prompt, decoding, and metrics. The final test partition remains untouched.

## Q&A

**Question:** How can GRPO fail even with correctly calculated rewards?  
**Answer:** Uniform groups have no relative signal, while a narrow or exploitable reward can encourage shortcuts or overfit the training questions. Independent validation must measure biomedical decisions, rather than reusing training rewards as the only outcome.